## Ejercicio Time Series Forecast
Para este ejercicio vamos a predecir cuál será la demanda de pasajeros de una aerolinea, para poder anticiparse a las contrataciones de personal, mantenimiento de las aeronaves y gestión de inventario y comidas.

Para ello, se pide:
1. Carga datos (AirPassengers.csv) y representa la serie. ¿Hay seasonality? ¿Cada cuanto?
2. Crea en una gráfica la variable original + su media obtenida mediante una rolling window con el valor de seasonality obtenido en el apartado anterior. Tienes que usar la función rolling() del DataFrame.
3. Comprueba de manera estadística si la serie es o no stationary.
4. Aplica una transformación logarítmica sobre los datos para mejorar el proceso de transformación de tu time series a stationary. Acuérdate después del forecast de invertir la transformación.
5. Divide en train y test. Guarda 20 muestras para test.
6. Crea tu primer modelo ARIMA. Habrá varias combinaciones en función de sus hiperparámetros... Mide el MAE y RMSE del modelo en predicción. Ten en cuenta el parámetro "m" de la función ARIMA, mediante el cual se establece el seasonality.
7. Representa en una gráfica los datos de test y tus predicciones.
8. Prueba un decission tree y un random forest, a ver qué performance presentan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

## 1. Carga datos y representa la serie

In [ ]:
df = pd.read_csv('data/AirPassengers.csv', parse_dates=True, index_col=0)
df.index = pd.DatetimeIndex(df.index).to_period('M')
df.head()

In [ ]:
df.info()

In [ ]:
df.head(20)

In [ ]:
df.plot(figsize=(16,5))
plt.show()

In [ ]:
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df)
plt.show()

## 2. Crea en una gráfica la variable original + su media obtenida mediante una rolling window con el valor de seasonality obtenido en el apartado anterior

In [ ]:
df['rolling_mean_12'] = df['value'].rolling(window=12).mean()

df[['value', 'rolling_mean_12']].plot(figsize=(16,5))
plt.show()

## 3. Comprueba de manera estadística si la serie es o no stationary.

El test estadístico es positivo, lo cual implica que es mucho menos probable que rechacemos la hipótesis nula (no estacionaria).

Al comparar el estadístico ADF con los valores críticos, parece que no podríamos rechazar la hipótesis nula de que la serie temporal no es estacionaria y en consecuencia afirmamos que la serie tiene una estructura que sí que es dependiente del tiempo.

Un valor p por encima del umbral sugiere que no rechazamos la hipótesis nula (no estacionario).

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_test = adfuller(df['value'])
print(f'ADF Statistic: {adf_test[0]:.6f}')
print(f'p-value: {adf_test[1]:.6f}')
print('Critical Values:')
for key, value in adf_test[4].items():
    print(f'\t{key}: {value:.3f}')

## 4. Aplica una transformación logarítmica

Podemos ver que el valor es mayor que los valores críticos, lo que significa que podemos rechazar la hipótesis nula y, a su vez, que la serie de tiempo no es estacionaria.

Sin embargo nos sigue interesando aplicar la transformación porque conseguimos estabilizar la varianza.

In [ ]:
df['log_value'] = np.log(df['value'])
adf_test_log = adfuller(df['log_value'])
print(f'ADF Statistic: {adf_test_log[0]:.6f}')
print(f'p-value: {adf_test_log[1]:.6f}')
print('Critical Values:')
for key, value in adf_test_log[4].items():
    print(f'\t{key}: {value:.3f}')

In [ ]:
df['log_value'].plot(figsize=(16,5))
plt.show()

## 5. Divide en train y test. Guarda 20 muestras para test.

In [ ]:
train = df['log_value'][:-20]
test = df['log_value'][-20:]

## 6. Crea tu primer modelo ARIMA

In [ ]:
import pmdarima as pm

model = pm.auto_arima(train, seasonal=False, stepwise=True, suppress_warnings=True, trace=True)
print(model.aic())

In [ ]:
model_seasonal = pm.auto_arima(train, seasonal=True, m=12, stepwise=True, suppress_warnings=True, trace=True)
print(model_seasonal.aic())

In [ ]:
model_seasonal.summary()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

predictions = model_seasonal.predict(n_periods=len(test))
predictions = pd.Series(predictions, index=test.index)

mse = mean_squared_error(test, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test, predictions)

print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'MAE: {mae}')

## 7. Representa en una gráfica los datos de test y tus predicciones.

In [ ]:
plt.figure(figsize=(16,5))
plt.plot(train.index, train, label='Train')
plt.plot(test.index, test, label='Test')
plt.plot(predictions.index, predictions, label='Predictions')
plt.legend()
plt.show()

In [ ]:
test

## 8. Prueba otros modelos, a ver qué performance presentan.

In [ ]:
df['t-1'] = df['log_value'].shift(1)
df['t-2'] = df['log_value'].shift(2)
df['t-3'] = df['log_value'].shift(3)
df['t-4'] = df['log_value'].shift(4)
df['t-5'] = df['log_value'].shift(5)
df['t-6'] = df['log_value'].shift(6)
df['t-7'] = df['log_value'].shift(7)
df['t-8'] = df['log_value'].shift(8)
df['t-9'] = df['log_value'].shift(9)
df['t-10'] = df['log_value'].shift(10)
df['t-11'] = df['log_value'].shift(11)
df['t-12'] = df['log_value'].shift(12)
df = df.dropna()

In [ ]:
X = df.drop('value', axis=1)
y = df['log_value']
X_train, X_test, y_train, y_test = X[:-20], X[-20:], y[:-20], y[-20:]
print(f'Shape X_train: {X_train.shape}')
print(f'Shape X_test: {X_test.shape}')
print(f'Shape y_train: {y_train.shape}')
print(f'Shape y_test: {y_test.shape}')

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
mae_dt = mean_absolute_error(y_test, y_pred_dt)
print(f'MSE: {mse_dt}')
print(f'RMSE: {rmse_dt}')
print(f'MAE: {mae_dt}')

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

plt.figure(figsize=(16,5))
plt.plot(y_test.index, y_test, label='Test')
plt.plot(y_test.index, y_pred_dt, label='Decision Tree')
plt.plot(y_test.index, y_pred_rf, label='Random Forest')
plt.legend()
plt.show()